<a href="https://colab.research.google.com/github/atulpatel12/Build-an-IMDB-sentiment-classifier-using-BERT-with-Gradio-UI/blob/main/Build_an_IMDB_sentiment_classifier_using_BERT_with_Gradio_Ui_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

What will we do :
1. Install libraries and set up
2. Load the IMDB dataset
3. Look at the data
4. Split into Train,Validation and Test
5. Tokenize with the BERT tokenizer
6. Load Pre-Trained BERT
7. Fine-Tune with Huggine-Face Trainer
8. Evaluate on the test set
9. Predict sentiment on new reviews
10. Save the model
11. Create a Gradio UI





In [ ]:
!pip install -q transformers datasets scikit-learn accelerate gradio

Importing Required Libraries

In [ ]:
import random
import numpy as np
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
   torch.cuda.manual_seed(SEED)

Load IMDB Dataset
we load IMDB dataset from Hugging Face's datasets hub ,It comes in pre split into train and test

In [ ]:
dataset = load_dataset("stanfordnlp/imdb")

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/unsupervised-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 42.0MB            

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


Look at a sample

Each sample has two flags


1.   Review: The movie review
2.   label : 0->Negative 1-> Positive



In [ ]:
sample = dataset["train"][0]
print("Label",sample["label"],"(0= negative,1 = positive)")
print("\nReview",sample["text"][:500],"...")

Label 0 (0= negative,1 = positive)

Review I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attent ...


Take a small subset(for fast training)

In [ ]:
TRAIN_SIZE = 2000
TEST_SIZE = 500

#we shuffle the dataset train and test so that the model radomly learn the negative as well as the positive review.. it might be possible that the first 2000 data we are takin
#might be all positive or negative

small_train = dataset["train"].shuffle(seed = SEED).select(range(TRAIN_SIZE))
small_test = dataset["train"].shuffle(seed = SEED).select(range(TRAIN_SIZE))

#split the training dataset int train + validation
split= small_train.train_test_split(test_size=0.1,seed = SEED)
train_ds = split["train"]
val_ds = split["test"]

print("Train size",len(train_ds))
print("Val size",len(val_ds))
print("Test size",len(small_test))

Train size 1800
Val size 200
Test size 2000


Tokenize The Text

In [ ]:
MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 256
# a tokenizer is a technique that tokenizes the large sentences into small words
# or tokens
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
  return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

# tokenize all three sets (unindented so they execute)
train_tokenized = train_ds.map(tokenize, batched=True)
val_tokenized = val_ds.map(tokenize, batched=True)
test_tokenized = small_test.map(tokenize, batched=True)

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Load pre-trained BERT for Classification

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels = 2
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Metrics and Data Collator

1.   Metrics : accuracy and F1 score ,computed after each epoch
2.   Data collator : pads each batch to the length of the longest review in the batch (more effecient than padding eveything to 256)

In [ ]:
def compute_metrics(eval_pred):
  logits,labels = eval_pred
  preds = np.argmax(logits,axis = 1)
  return {
      "accuracy" : accuracy_score(labels,preds),
      "f1": f1_score(labels,preds)
  }

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer = tokenizer)

Set Up Trainer

In [ ]:
training_args = TrainingArguments(
    output_dir ="/bert_imdb_output",
    eval_strategy ="epoch",
    learning_rate = 2e-5,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size= 8,
    num_train_epochs=2,
    weight_decay=0.01,
    fp16 =torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
    logging_steps=50,
)
trainer= Trainer(
    model= model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

Fine-Tuning The Model

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.094791,0.871362,0.835000,0.835821
2,0.065393,0.573823,0.895000,0.888889


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=450, training_loss=0.10459685431586371, metrics={'train_runtime': 123.5781, 'train_samples_per_second': 29.131, 'train_steps_per_second': 3.641, 'total_flos': 473271010828800.0, 'train_loss': 0.10459685431586371, 'epoch': 2.0})

Evaluate on test-set

In [ ]:
final_results = trainer.evaluate(eval_dataset=test_tokenized)
print("final results:")
for k, v in final_results.items():
  if isinstance(v, float):
    print(f" {k}: {v:.4f}")

Training Loss,Validation Loss,Epoch,Accuracy,F1
No log,0.089727,0,0.983500,0.983508


final results:
 eval_loss: 0.0897
 eval_accuracy: 0.9835
 eval_f1: 0.9835


Predict sentiments on new reviews

In [ ]:
id2label = {0: "negative", 1: "positive"}

def predict_sentiments(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    ).to(model.device)

    model.eval()
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)[0]
        pred_id = torch.argmax(probs).item()

        return {
            "label": id2label[pred_id],
            "confidence": round(probs[pred_id].item(), 4),
        }

# Try it out
reviews = [
    "This movie was absolutely fantastic. Brilliant acting and a gripping story",
    "I hated this film. It was boring, too long, and badly written ",
    "It was okay - some good moments, but overall forgettable"
]

for r in reviews:
    print(predict_sentiments(r), "->", r[:60], "...")

{'label': 'positive', 'confidence': 0.9994} -> This movie was absolutely fantastic. Brilliant acting and a  ...
{'label': 'negative', 'confidence': 0.998} -> I hated this film. It was boring, too long, and badly writte ...
{'label': 'positive', 'confidence': 0.9177} -> It was okay - some good moments, but overall forgettable ...


Save the Model and Tokenizer

In [ ]:
SAVE_DIR ="./imdb_bert"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print("Saved model and tokenizer to:", SAVE_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model and tokenizer to: ./imdb_bert


Gradio UI

In [ ]:
import torch
import gradio as gr

from transformers import AutoTokenizer, AutoModelForSequenceClassification
# -------------Reload from saved directory------------
SAVE_DIR = "./imdb_bert"

reloaded_tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
reloaded_model = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR)

device = "cuda" if torch.cuda.is_available() else "cpu"
reloaded_model.to(device)
reloaded_model.eval()

print("Model and tokenizer reloaded on:", device)

#-------- Interface Function-------------
id2label = {0: "negative", 1: "positive"}
MAX_LENGTH = 256

def predict_sentiment(text):
    inputs = reloaded_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    ).to(device)

    with torch.no_grad():
        logits = reloaded_model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)[0]
        pred_id = torch.argmax(probs).item()
        return id2label[pred_id], probs[pred_id].item()

# -------Gradio UI ---------------
def classify_review(review):
    if not review.strip():
        return "Please enter a review."
    label, confidence = predict_sentiment(review)
    emoji = "✅" if label == "positive" else "❌"
    return f"{emoji} {label} ({confidence*100:.2f}% confidence)"

demo = gr.Interface(
    fn=classify_review,
    inputs=gr.Textbox(
        lines=5,
        label="Movie Review",
        placeholder="Type a movie review here.."
    ),
    outputs=gr.Markdown(label="Prediction"),
    title="IMDB Sentiment Classifier",
    description="Fine-tuning BERT predicting whether a movie review is positive or negative.",
    flagging_mode="never",
    examples=[
        ["This movie was absolutely fantastic. Brilliant acting and a gripping story"],
        ["I hated this film. It was boring, too long, and badly written "],
        ["It was okay - some good moments, but overall forgettable"]
    ]
)

demo.launch(share=True)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model and tokenizer reloaded on: cuda
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://89d3c80d4362b072f8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
